# LeetCode #199 — Binary Tree Right Side View: nút cuối mỗi tầng (BFS) hoặc DFS ưu tiên bên phải

**Mức độ:** Medium · **Chủ đề:** Tree, Depth-First Search, Breadth-First Search, Binary Tree

Skeleton trên LeetCode:

```python
class Solution:
    def rightSideView(self, root: Optional[TreeNode]) -> List[int]:
```

Mục tiêu của notebook:

1. Hiểu "nhìn từ bên phải": ở **mỗi tầng** chỉ thấy nút **ngoài cùng bên phải** — không phải "đi theo con phải".
2. Cách 1: **BFS theo tầng**, lấy phần tử cuối mỗi tầng.
3. Cách 2: **DFS** thăm con **phải trước**; nút **đầu tiên** gặp ở độ sâu `d` là nút nhìn thấy.
4. Biến thể: nhìn từ trái, nhìn từ trên.

# 1. Bài toán là gì?

Cho `root` của cây nhị phân, tưởng tượng bạn đứng ở **bên phải** cây. Trả về giá trị các nút bạn **nhìn thấy**, từ trên xuống dưới.

**Constraints (theo đề):**

- Số nút trong khoảng `[0, 100]`.
- `-100 <= Node.val <= 100`

---

## Ví dụ

```text
[1, 2, 3, null, 5, null, 4]       ->  [1, 3, 4]

        1            ← thấy 1
       / \
      2   3          ← thấy 3
       \   \
        5   4        ← thấy 4 (5 bị 4 che)

[1, 2, 3, 4, null, null, null, 5] ->  [1, 3, 4, 5]

        1
       / \
      2   3          ← thấy 3
     /
    4                ← thấy 4 (tầng này không có nút nào bên phải 4)
   /
  5                  ← thấy 5

[1, null, 3] -> [1, 3]          []  ->  []
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân (binary tree)
2. Duyệt theo chiều rộng (BFS — Breadth-First Search)
3. Duyệt theo chiều sâu (DFS — Depth-First Search)

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân (binary tree)

### Cây nhị phân là gì?

Cây nhị phân gồm các **nút**; mỗi nút có một giá trị và tối đa **hai con**: con **trái** và con **phải**. Nút trên cùng là **gốc** (root); nút không có con là **lá** (leaf).

```text
          1          ← gốc (độ sâu 0)
        /   \
       2     3       ← độ sâu 1
      / \     \
     4   5     6     ← lá 4, 5, 6 (độ sâu 2)
```

Thuật ngữ: **cha/con**, **anh em** (cùng cha), **cây con** (một nút cùng toàn bộ con cháu), **độ sâu** (số cạnh từ gốc), **chiều cao** (số tầng / đường dài nhất từ gốc tới lá).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Dựng cây ở hình trên bằng tay
goc = TreeNode(1,
               TreeNode(2, TreeNode(4), TreeNode(5)),
               TreeNode(3, None, TreeNode(6)))
print("gốc:", goc.val, "| con trái:", goc.left.val, "| con phải:", goc.right.val,
      "| 3 có con trái?", goc.right.left)

gốc: 1 | con trái: 2 | con phải: 3 | 3 có con trái? None


### Biểu diễn kiểu LeetCode: danh sách theo tầng (level-order)

LeetCode ghi cây thành danh sách đọc từng tầng, từ trái sang phải, `None` cho chỗ trống: cây trên là `[1, 2, 3, 4, 5, None, 6]`. Dựng cây từ danh sách này bằng một **hàng đợi**: lần lượt lấy nút cha ra, gán hai giá trị kế tiếp làm con trái/phải.

In [2]:
from collections import deque

def tao_cay(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def ve_cay(node, pre="", la_phai=True, goc_cay=True):
    """In cây nằm ngang để dễ nhìn."""
    if node is None:
        return
    ve_cay(node.right, pre + ("" if goc_cay else ("    " if la_phai else "│   ")), True, False)
    print(pre + ("" if goc_cay else ("┌── " if la_phai else "└── ")) + str(node.val))
    ve_cay(node.left, pre + ("" if goc_cay else ("│   " if la_phai else "    ")), False, False)

ve_cay(tao_cay([1, 2, 3, 4, 5, None, 6]))

    ┌── 6
┌── 3
1
│   ┌── 5
└── 2
    └── 4


(Cây in nằm ngang: gốc ở bên trái, con **phải** ở phía **trên**, con **trái** ở phía **dưới**.)

### Ba cách duyệt theo chiều sâu

Với mỗi nút ta làm ba việc: thăm **nút**, duyệt cây con **trái**, duyệt cây con **phải**. Thứ tự của ba việc cho ba kiểu duyệt:

| Kiểu | Thứ tự | Cây trên |
|---|---|---|
| **Preorder** (tiền thứ tự) | nút → trái → phải | `1 2 4 5 3 6` |
| **Inorder** (trung thứ tự) | trái → nút → phải | `4 2 5 1 3 6` |
| **Postorder** (hậu thứ tự) | trái → phải → nút | `4 5 2 6 3 1` |

Mẹo nhớ: "pre/in/post" chỉ **vị trí của nút** so với hai cây con.

In [3]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)        # nút
    preorder(node.left, out)    # trái
    preorder(node.right, out)   # phải

def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)

def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)

cay = tao_cay([1, 2, 3, 4, 5, None, 6])
for ten, f in [("preorder", preorder), ("inorder", inorder), ("postorder", postorder)]:
    out = []; f(cay, out); print(f"{ten:<9}:", out)

preorder : [1, 2, 4, 5, 3, 6]
inorder  : [4, 2, 5, 1, 3, 6]
postorder: [4, 5, 2, 6, 3, 1]


### Duyệt không đệ quy bằng ngăn xếp

Đệ quy thực chất dùng *call stack*; ta có thể tự quản lý ngăn xếp. Preorder: lấy nút ra, ghi lại, đẩy con **phải trước** rồi con **trái** (để trái được lấy ra trước). Inorder: đi trái hết mức, lấy ra, rồi sang phải.

In [4]:
def preorder_lap(root):
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node.val)
        if node.right: st.append(node.right)
        if node.left:  st.append(node.left)
    return out

def inorder_lap(root):
    out, st, cur = [], [], root
    while cur or st:
        while cur:              # đi trái hết mức
            st.append(cur); cur = cur.left
        cur = st.pop()
        out.append(cur.val)
        cur = cur.right
    return out

print("preorder lặp:", preorder_lap(cay), "| inorder lặp:", inorder_lap(cay))

preorder lặp: [1, 2, 4, 5, 3, 6] | inorder lặp: [4, 2, 5, 1, 3, 6]


### Tư duy đệ quy trên cây

Hầu hết bài về cây giải bằng cách hỏi: "nếu đã biết đáp án cho **cây con trái** và **cây con phải**, ghép lại thế nào?". Ví dụ chiều cao, số nút, tổng giá trị:

In [5]:
def chieu_cao(node):
    if node is None:
        return 0                                   # cây rỗng cao 0
    return 1 + max(chieu_cao(node.left), chieu_cao(node.right))

def so_nut(node):
    return 0 if node is None else 1 + so_nut(node.left) + so_nut(node.right)

print("chiều cao:", chieu_cao(cay), "| số nút:", so_nut(cay))

chiều cao: 3 | số nút: 6


### Lỗi hay gặp

- Quên trường hợp `node is None` → `AttributeError: 'NoneType' object has no attribute 'left'`.
- Nhầm "lá" (không có con nào) với "nút thiếu một con".
- Nhầm thứ tự preorder/inorder/postorder.
- Cây lệch (như danh sách) cao tới `n` → đệ quy sâu `n` tầng.

## 2.2 Duyệt theo chiều rộng (BFS — Breadth-First Search)

### BFS là gì?

BFS duyệt đồ thị/cây **theo từng lớp**: thăm điểm xuất phát, rồi **mọi** điểm cách 1 bước, rồi mọi điểm cách 2 bước, … giống vết sóng lan ra khi ném hòn đá xuống nước.

Công cụ: **hàng đợi** (FIFO). Điểm phát hiện trước được xử lý trước → các điểm gần luôn được xử lý trước các điểm xa.

```text
        A            lớp 0: A
       / \           lớp 1: B, C
      B   C          lớp 2: D, E, F
     / \   \
    D   E   F        thứ tự BFS: A B C D E F
```

### Khung BFS trên đồ thị

```text
q = deque([start]); visited = {start}
while q:
    u = q.popleft()
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            visited.add(v)        # đánh dấu NGAY khi đưa vào hàng đợi
            q.append(v)
```

In [6]:
from collections import deque

do_thi = {                       # danh sách kề
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def bfs(graph, start):
    q, visited, thu_tu = deque([start]), {start}, []
    while q:
        u = q.popleft()
        thu_tu.append(u)
        for v in graph[u]:
            if v not in visited:
                visited.add(v)
                q.append(v)
        print(f"lấy {u} | hàng đợi: {list(q)}")
    return thu_tu

print("Thứ tự BFS:", bfs(do_thi, "A"))

lấy A | hàng đợi: ['B', 'C']
lấy B | hàng đợi: ['C', 'D', 'E']
lấy C | hàng đợi: ['D', 'E', 'F']
lấy D | hàng đợi: ['E', 'F']
lấy E | hàng đợi: ['F']
lấy F | hàng đợi: []
Thứ tự BFS: ['A', 'B', 'C', 'D', 'E', 'F']


### BFS theo tầng (level-order) trên cây nhị phân

Muốn biết nút nào thuộc tầng nào: ở đầu mỗi vòng, hàng đợi chứa **đúng** các nút của một tầng. Ghi lại `len(q)`, lấy ra đúng chừng ấy nút (và đẩy con của chúng vào) — đó là một tầng.

In [7]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_bfs = TreeNode(3, TreeNode(9), TreeNode(20, TreeNode(15), TreeNode(7)))

def theo_tang(root):
    if not root:
        return []
    q, tang = deque([root]), []
    while q:
        cur = []
        for _ in range(len(q)):          # đúng số nút của tầng hiện tại
            node = q.popleft()
            cur.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        tang.append(cur)
    return tang

print("các tầng:", theo_tang(cay_bfs))

các tầng: [[3], [9, 20], [15, 7]]


### BFS tìm đường ngắn nhất trên lưới

Trên đồ thị **không trọng số** (mỗi bước dài như nhau), lần đầu BFS chạm tới một ô chính là đi bằng **số bước ít nhất**. Lưu `dist` khi đưa ô vào hàng đợi.

In [8]:
luoi = ["S.#.",
        "..#.",
        "...E"]

def ngan_nhat(grid):
    R, C = len(grid), len(grid[0])
    start = next((r, c) for r in range(R) for c in range(C) if grid[r][c] == "S")
    dist = {start: 0}
    q = deque([start])
    while q:
        r, c = q.popleft()
        if grid[r][c] == "E":
            return dist[(r, c)]
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                q.append((nr, nc))
    return -1

print("số bước ít nhất từ S tới E:", ngan_nhat(luoi))

số bước ít nhất từ S tới E: 5


### Độ phức tạp và lỗi hay gặp

`O(V + E)` thời gian (mỗi đỉnh vào hàng đợi một lần, mỗi cạnh xét một/hai lần), `O(V)` bộ nhớ. Trên cây: `O(n)`; trên lưới `R×C`: `O(R·C)`.

- Đánh dấu `visited` khi **lấy ra** thay vì khi **đưa vào** → một ô có thể vào hàng đợi nhiều lần (chậm, đếm sai).
- Dùng `list.pop(0)` → mỗi lần `O(n)`.
- Dùng `pop()` (cuối) → thành DFS, mất tính "gần trước".
- Quên kiểm tra biên lưới.

## 2.3 Duyệt theo chiều sâu (DFS — Depth-First Search)

### DFS là gì?

DFS đi **sâu nhất có thể** theo một nhánh, tới khi hết đường thì **quay lui** về ngã rẽ gần nhất và thử nhánh khác — giống đi mê cung bằng cách luôn bám một bức tường.

```text
        A
       / \          DFS từ A (thử hàng xóm theo thứ tự):
      B   C         A → B → D (cụt, quay lui) → E → F → C
     / \   \
    D   E - F
```

Công cụ: **đệ quy** (dùng call stack) hoặc **ngăn xếp** tự quản lý.

### Khung DFS

```text
def dfs(u):
    visited.add(u)
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            dfs(v)
```

In [9]:
do_thi_dfs = {
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def dfs_de_quy(graph, u, visited, thu_tu, sau=0):
    visited.add(u)
    thu_tu.append(u)
    print("    " * sau + f"vào {u}")
    for v in graph[u]:
        if v not in visited:
            dfs_de_quy(graph, v, visited, thu_tu, sau + 1)
    print("    " * sau + f"rời {u}")

tt = []
dfs_de_quy(do_thi_dfs, "A", set(), tt)
print("Thứ tự DFS:", tt)

vào A
    vào B
        vào D
        rời D
        vào E
            vào F
                vào C
                rời C
            rời F
        rời E
    rời B
rời A
Thứ tự DFS: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS bằng ngăn xếp (không đệ quy)

Thay hàng đợi của BFS bằng **ngăn xếp**. Tránh được giới hạn độ sâu đệ quy khi dữ liệu lớn.

In [10]:
def dfs_ngan_xep(graph, start):
    st, visited, thu_tu = [start], set(), []
    while st:
        u = st.pop()
        if u in visited:
            continue
        visited.add(u)
        thu_tu.append(u)
        for v in reversed(graph[u]):        # đảo để hàng xóm đầu tiên được lấy ra trước
            if v not in visited:
                st.append(v)
    return thu_tu

print("DFS bằng ngăn xếp:", dfs_ngan_xep(do_thi_dfs, "A"))

DFS bằng ngăn xếp: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS trên lưới: tô loang (flood fill) và đếm vùng

Lưới là đồ thị ẩn: mỗi ô nối với 4 ô kề. DFS từ một ô sẽ "loang" khắp vùng liên thông chứa nó. Đếm số lần phải bắt đầu DFS mới = số vùng.

In [11]:
def dem_dao(grid):
    g = [list(row) for row in grid]
    R, C = len(g), len(g[0])
    def loang(r, c):
        if r < 0 or r >= R or c < 0 or c >= C or g[r][c] != "1":
            return                              # ra ngoài / nước / đã thăm
        g[r][c] = "#"                           # đánh dấu đã thăm
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            loang(r + dr, c + dc)
    dem = 0
    for r in range(R):
        for c in range(C):
            if g[r][c] == "1":
                dem += 1
                loang(r, c)
    return dem, ["".join(row) for row in g]

print(dem_dao(["11000",
               "11010",
               "00011"]))

(2, ['##000', '##0#0', '000##'])


### DFS trên cây: mang thông tin dọc đường đi

Trên cây không cần `visited` (không có chu trình). DFS hay **truyền xuống** thông tin của đường đi từ gốc (tổng, đường đi hiện tại) và **trả lên** kết quả của cây con.

In [12]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_dfs = TreeNode(5, TreeNode(4, TreeNode(11, TreeNode(7), TreeNode(2))), TreeNode(8, TreeNode(13), TreeNode(4)))

def moi_duong_goc_la(node, duong, out):
    if node is None:
        return
    duong.append(node.val)                     # đi xuống: thêm nút
    if node.left is None and node.right is None:
        out.append((list(duong), sum(duong)))  # tới lá: ghi lại một bản sao
    moi_duong_goc_la(node.left, duong, out)
    moi_duong_goc_la(node.right, duong, out)
    duong.pop()                                # quay lên: bỏ nút

ket_qua = []
moi_duong_goc_la(cay_dfs, [], ket_qua)
for d, s in ket_qua:
    print(d, "tổng =", s)

[5, 4, 11, 7] tổng = 27
[5, 4, 11, 2] tổng = 22
[5, 8, 13] tổng = 26
[5, 8, 4] tổng = 17


### So sánh BFS và DFS

| | BFS | DFS |
|---|---|---|
| Cấu trúc | hàng đợi | ngăn xếp / đệ quy |
| Thứ tự | theo lớp (gần → xa) | theo nhánh (sâu trước) |
| Đường ngắn nhất (không trọng số) | **có** | không đảm bảo |
| Bộ nhớ | độ rộng lớn nhất | độ sâu lớn nhất |
| Hợp với | khoảng cách, theo tầng | liệt kê đường đi, vùng liên thông, quay lui |

Lỗi hay gặp: quên `visited` trên đồ thị có chu trình (lặp vô hạn); đệ quy quá sâu trên lưới lớn; ghi `out.append(duong)` thay vì bản sao `list(duong)` (về sau list bị sửa).

## 2.4 Áp dụng vào bài này

"Nhìn từ bên phải" = nút **cuối cùng của mỗi tầng**. **BFS theo tầng** (xử lý trọn `len(queue)` nút mỗi vòng) lấy nút cuối mỗi tầng; hoặc **DFS** ưu tiên con **phải** trước, ghi nút đầu tiên gặp ở mỗi độ sâu (`depth == len(result)`).

# 3. Nền tảng

In [13]:
from typing import List, Optional
from collections import deque
import random


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root


def levels(root):
    out, q = [], deque([root] if root else [])
    while q:
        cur = []
        for _ in range(len(q)):
            node = q.popleft()
            cur.append(node.val)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        out.append(cur)
    return out

print("các tầng của ví dụ 2:", levels(build_tree([1, 2, 3, 4, None, None, None, 5])))

các tầng của ví dụ 2: [[1], [2, 3], [4], [5]]


Nhìn từ bên phải = lấy **phần tử cuối** của mỗi tầng trong danh sách trên: `[1, 3, 4, 5]`.

# 4. Cách hiểu sai thường gặp: chỉ đi theo con phải

In [14]:
def follow_right_only(root):
    out = []
    while root:
        out.append(root.val)
        root = root.right
    return out

print("ví dụ 1:", follow_right_only(build_tree([1, 2, 3, None, 5, None, 4])), "(tình cờ đúng)")
print("ví dụ 2:", follow_right_only(build_tree([1, 2, 3, 4, None, None, None, 5])), "← thiếu 4, 5")

ví dụ 1: [1, 3, 4] (tình cờ đúng)
ví dụ 2: [1, 3] ← thiếu 4, 5


Khi nhánh phải **ngắn hơn** nhánh trái, các tầng sâu hơn vẫn có nút nhìn thấy được — chúng nằm ở nhánh **trái**.

# 5. Cách 1: BFS theo tầng, lấy phần tử cuối

```text
q = [root]
while q:
    lấy đúng len(q) nút của tầng hiện tại; nút lấy ra CUỐI CÙNG là nút ngoài cùng bên phải
    đẩy con trái rồi con phải của chúng vào q
```

In [15]:
def right_view_bfs(root: Optional[TreeNode]) -> List[int]:
    if not root:
        return []
    out, q = [], deque([root])
    while q:
        size = len(q)
        for i in range(size):
            node = q.popleft()
            if i == size - 1:
                out.append(node.val)          # nút cuối của tầng
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
    return out

print(right_view_bfs(build_tree([1, 2, 3, None, 5, None, 4])), right_view_bfs(build_tree([1, 2, 3, 4, None, None, None, 5])))

[1, 3, 4]

 [1, 3, 4, 5]


# 6. Cách 2: DFS ưu tiên con phải

Duyệt **nút → phải → trái** (preorder "đảo"), mang theo độ sâu `d`. Vì luôn đi phải trước, **lần đầu** chạm tới độ sâu `d` chính là nút ngoài cùng bên phải của tầng `d`. Điều kiện "lần đầu": `d == len(out)`.

In [16]:
def right_view_dfs(root: Optional[TreeNode]) -> List[int]:
    out = []
    def dfs(node, d):
        if node is None:
            return
        if d == len(out):                     # lần đầu tới tầng d
            out.append(node.val)
        dfs(node.right, d + 1)                # phải trước
        dfs(node.left, d + 1)
    dfs(root, 0)
    return out

print(right_view_dfs(build_tree([1, 2, 3, None, 5, None, 4])), right_view_dfs(build_tree([1, 2, 3, 4, None, None, None, 5])))

[1, 3, 4] [1, 3, 4, 5]


# 7. Mô phỏng

In [17]:
def trace_dfs(values):
    out = []
    def dfs(node, d, ind=""):
        if node is None:
            return
        first = d == len(out)
        if first:
            out.append(node.val)
        print(f"{ind}thăm {node.val} (tầng {d}){' ← lần đầu tới tầng này, ghi lại' if first else ''}")
        dfs(node.right, d + 1, ind + "    ")
        dfs(node.left, d + 1, ind + "    ")
    dfs(build_tree(values), 0)
    print("→", out)

trace_dfs([1, 2, 3, 4, None, None, None, 5])

thăm 1 (tầng 0) ← lần đầu tới tầng này, ghi lại


    thăm 3 (tầng 1) ← lần đầu tới tầng này, ghi lại
    thăm 2 (tầng 1)
        thăm 4 (tầng 2) ← lần đầu tới tầng này, ghi lại
            thăm 5 (tầng 3) ← lần đầu tới tầng này, ghi lại
→ [1, 3, 4, 5]


# 8. Phiên bản đúng format LeetCode

In [18]:
class Solution:
    def rightSideView(self, root: Optional[TreeNode]) -> List[int]:
        if not root:
            return []
        out, q = [], deque([root])
        while q:
            out.append(q[-1].val)             # phần tử cuối của hàng đợi = nút phải nhất của tầng
            for _ in range(len(q)):
                node = q.popleft()
                if node.left:
                    q.append(node.left)
                if node.right:
                    q.append(node.right)
        return out

Mẹo gọn: ở đầu mỗi vòng, hàng đợi chứa đúng các nút của một tầng theo thứ tự trái → phải, nên `q[-1]` là nút phải nhất.

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<12} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.rightSideView(build_tree([1, 2, 3, None, 5, None, 4])), [1, 3, 4])
check("Ví dụ 2", sol.rightSideView(build_tree([1, 2, 3, 4, None, None, None, 5])), [1, 3, 4, 5])
check("Ví dụ 3", sol.rightSideView(build_tree([1, None, 3])), [1, 3])
check("Rỗng", sol.rightSideView(None), [])
check("Lệch trái", sol.rightSideView(build_tree([1, 2, None, 3])), [1, 2, 3])


def random_tree(n):
    if n == 0:
        return None
    nodes = [TreeNode(random.randint(-100, 100)) for _ in range(n)]
    for i in range(1, n):
        while True:
            p = nodes[random.randrange(i)]
            side = random.choice(["left", "right"])
            if getattr(p, side) is None:
                setattr(p, side, nodes[i]); break
    return nodes[0]


random.seed(199)
for _ in range(2000):
    r = random_tree(random.randint(0, 15))
    exp = [lv[-1] for lv in levels(r)]
    assert sol.rightSideView(r) == exp == right_view_bfs(r) == right_view_dfs(r)
print("2000 cây ngẫu nhiên: ba cách khớp 'phần tử cuối mỗi tầng'")

Ví dụ 1      PASS  [1, 3, 4]
Ví dụ 2      PASS  [1, 3, 4, 5]
Ví dụ 3      PASS  [1, 3]
Rỗng         PASS  []
Lệch trái    PASS  [1, 2, 3]


2000 cây ngẫu nhiên: ba cách khớp 'phần tử cuối mỗi tầng'


# 9. Vì sao đúng? Độ phức tạp?

- **BFS:** mỗi vòng `while` xử lý đúng một tầng, các nút được đưa vào hàng đợi theo thứ tự trái → phải (đẩy con trái trước con phải, và tầng trên đã theo thứ tự đó). Phần tử cuối là nút phải nhất.
- **DFS phải trước:** trong thứ tự thăm "nút → phải → trái", với mỗi tầng `d`, nút phải nhất của tầng được thăm **trước** mọi nút khác cùng tầng (mọi nút bên trái nó nằm trong cây con được thăm sau). Điều kiện `d == len(out)` ghi đúng nút đầu tiên ở mỗi tầng.

```text
Time:  O(n)
Space: BFS O(độ rộng lớn nhất); DFS O(h) ngăn xếp đệ quy
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Chỉ đi theo con phải (mục 4)

## Lỗi 2: DFS thăm **trái trước** mà vẫn dùng điều kiện "lần đầu"

In [20]:
def wrong_left_first(root):
    out = []
    def dfs(node, d):
        if node is None:
            return
        if d == len(out):
            out.append(node.val)
        dfs(node.left, d + 1)                      # SAI: trái trước → ghi nút TRÁI nhất
        dfs(node.right, d + 1)
    dfs(root, 0)
    return out

t2 = build_tree([1, 2, 3, None, 5, None, 4])
print("Sai :", wrong_left_first(t2), "| Đúng:", sol.rightSideView(t2))

Sai : [1, 2, 5] | Đúng: [1, 3, 4]


Kết quả lại là **nhìn từ bên trái**. Muốn trái trước thì phải **ghi đè** `out[d] = node.val` mỗi lần (nút thăm sau cùng ở tầng `d` là nút phải nhất).

## Lỗi 3: BFS lấy `q[-1]` **sau** khi đã đẩy con vào hàng đợi

In [21]:
def wrong_peek_late(root):
    if not root:
        return []
    out, q = [], deque([root])
    while q:
        for _ in range(len(q)):
            node = q.popleft()
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        if q:
            out.append(q[-1].val)                  # SAI: lúc này q là tầng KẾ TIẾP
    return out

print("Sai :", wrong_peek_late(t2), "| Đúng:", sol.rightSideView(t2))

Sai : [3, 4] | Đúng: [1, 3, 4]


Mất nút gốc (tầng 0) vì lần đọc đầu tiên đã là tầng 1.

# 11. Mẹo nhận diện pattern

- **"Theo tầng", "mỗi tầng một giá trị"** → BFS với `for _ in range(len(q))`.
- **"Nút đầu tiên/cuối cùng ở mỗi độ sâu"** → DFS mang độ sâu, ghi khi `d == len(out)` (đầu tiên) hoặc ghi đè (cuối cùng); chọn thứ tự thăm trái/phải cho phù hợp.

Bài liên quan: **#102 Binary Tree Level Order Traversal**, **#513 Find Bottom Left Tree Value**, **#515 Find Largest Value in Each Tree Row**, **#116/#117 Populating Next Right Pointers**, **#987 Vertical Order Traversal** (Hard).

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Viết các tầng và kết quả cho `[1, 2, 3, 4, 5, None, None, None, None, 6]`.

## Bài 2 — Điền chỗ trống (BFS)

In [22]:
def exercise_2_right_view(root: Optional[TreeNode]) -> List[int]:
    if not root:
        return []
    out, q = [], deque([root])
    while q:
        # TODO: ghi nút phải nhất của tầng, rồi xử lý cả tầng
        pass
    return out


# assert exercise_2_right_view(build_tree([1, 2, 3, None, 5, None, 4])) == [1, 3, 4]
# assert exercise_2_right_view(build_tree([1, 2, 3, 4, None, None, None, 5])) == [1, 3, 4, 5]
# assert exercise_2_right_view(None) == []

## Bài 3 — Viết bản DFS (phải trước)

In [23]:
def exercise_3_right_view_dfs(root: Optional[TreeNode]) -> List[int]:
    # TODO
    pass


# assert exercise_3_right_view_dfs(build_tree([1, 2, 3, None, 5, None, 4])) == [1, 3, 4]
# assert exercise_3_right_view_dfs(build_tree([1, 2, None, 3])) == [1, 2, 3]

## Bài 4 — Biến thể: nhìn từ bên **trái**

In [24]:
def exercise_4_left_view(root: Optional[TreeNode]) -> List[int]:
    # TODO
    pass


# assert exercise_4_left_view(build_tree([1, 2, 3, None, 5, None, 4])) == [1, 2, 5]
# assert exercise_4_left_view(build_tree([1, None, 3, None, 4])) == [1, 3, 4]

## Bài 5 — Mini challenge: nhìn từ **trên xuống** (top view)

Mỗi nút có **cột ngang**: gốc ở cột 0, con trái `cột − 1`, con phải `cột + 1`. Nhìn từ trên, ở mỗi cột thấy nút **cao nhất** (tầng nhỏ nhất); nếu cùng tầng, lấy nút gặp **trước** trong BFS (bên trái hơn). Trả về giá trị theo cột từ trái sang phải.

In [25]:
def exercise_5_top_view(root: Optional[TreeNode]) -> List[int]:
    # TODO
    pass


# assert exercise_5_top_view(build_tree([1, 2, 3, 4, 5, 6, 7])) == [4, 2, 1, 3, 7]
# assert exercise_5_top_view(build_tree([1, 2, 3, None, 4, None, None, None, 5, None, 6])) == [2, 1, 3, 6]
# assert exercise_5_top_view(None) == []

# 13. Gợi ý

**Bài 1**
- Mức 1: in `levels(...)` và chạy `trace_dfs` sau khi tự làm.

**Bài 4**
- Mức 1: BFS lấy `q[0]` (phần tử đầu) mỗi tầng; hoặc DFS trái trước với điều kiện "lần đầu".

**Bài 5**
- Mức 1: BFS với hàng đợi chứa `(nút, cột)`; dict `cột → giá trị`, chỉ ghi khi cột **chưa có** (BFS đảm bảo tầng nhỏ hơn đến trước).
- Mức 2: kết quả: `[d[c] for c in sorted(d)]`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [26]:
v1 = [1, 2, 3, 4, 5, None, None, None, None, 6]
print("tầng:", levels(build_tree(v1)))
trace_dfs(v1)

tầng: [[1], [2, 3], [4, 5], [6]]
thăm 1 (tầng 0) ← lần đầu tới tầng này, ghi lại
    thăm 3 (tầng 1) ← lần đầu tới tầng này, ghi lại
    thăm 2 (tầng 1)
        thăm 5 (tầng 2) ← lần đầu tới tầng này, ghi lại
            thăm 6 (tầng 3) ← lần đầu tới tầng này, ghi lại
        thăm 4 (tầng 2)
→ [1, 3, 5, 6]


## Đáp án Bài 2

In [27]:
def answer_2_right_view(root: Optional[TreeNode]) -> List[int]:
    return Solution().rightSideView(root)

assert answer_2_right_view(build_tree([1, 2, 3, None, 5, None, 4])) == [1, 3, 4]
assert answer_2_right_view(build_tree([1, 2, 3, 4, None, None, None, 5])) == [1, 3, 4, 5]
assert answer_2_right_view(None) == []
print("Bài 2 PASS (giống Solution ở mục 8)")

Bài 2 PASS (giống Solution ở mục 8)


## Đáp án Bài 3

In [28]:
def answer_3_right_view_dfs(root: Optional[TreeNode]) -> List[int]:
    return right_view_dfs(root)

assert answer_3_right_view_dfs(build_tree([1, 2, 3, None, 5, None, 4])) == [1, 3, 4]
assert answer_3_right_view_dfs(build_tree([1, 2, None, 3])) == [1, 2, 3]
print("Bài 3 PASS (giống right_view_dfs ở mục 6)")

Bài 3 PASS (giống right_view_dfs ở mục 6)


## Đáp án Bài 4

In [29]:
def answer_4_left_view(root: Optional[TreeNode]) -> List[int]:
    if not root:
        return []
    out, q = [], deque([root])
    while q:
        out.append(q[0].val)
        for _ in range(len(q)):
            node = q.popleft()
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
    return out

assert answer_4_left_view(build_tree([1, 2, 3, None, 5, None, 4])) == [1, 2, 5]
assert answer_4_left_view(build_tree([1, None, 3, None, 4])) == [1, 3, 4]
for _ in range(1000):
    r = random_tree(random.randint(0, 12))
    assert answer_4_left_view(r) == [lv[0] for lv in levels(r)]
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [30]:
def answer_5_top_view(root: Optional[TreeNode]) -> List[int]:
    if not root:
        return []
    first = {}
    q = deque([(root, 0)])
    while q:
        node, col = q.popleft()
        if col not in first:
            first[col] = node.val
        if node.left: q.append((node.left, col - 1))
        if node.right: q.append((node.right, col + 1))
    return [first[c] for c in sorted(first)]

assert answer_5_top_view(build_tree([1, 2, 3, 4, 5, 6, 7])) == [4, 2, 1, 3, 7]
assert answer_5_top_view(build_tree([1, 2, 3, None, 4, None, None, None, 5, None, 6])) == [2, 1, 3, 6]
assert answer_5_top_view(None) == []
print("Bài 5 PASS")

Bài 5 PASS

Ở ví dụ thứ hai, nút `4` (cột 0, tầng 2) bị gốc `1` (cột 0) che; nút `5` (cột 1) bị `3` che; còn `6` ở cột 2 thì không ai che.

# 15. Tổng kết

> Nhìn từ phải = **nút cuối của mỗi tầng**. BFS: đầu mỗi vòng ghi `q[-1].val` rồi xử lý cả tầng. DFS: thăm phải trước, ghi khi `d == len(out)`. Đừng chỉ đi theo con phải — nhánh trái dài hơn vẫn lộ ra ở các tầng sâu.

---

## Checklist trước khi nộp

- [ ] Lấy nút cuối **mỗi tầng**, không chỉ đi theo `right`?
- [ ] BFS: đọc `q[-1]` khi hàng đợi đang chứa đúng tầng hiện tại?
- [ ] DFS: phải trước trái?
- [ ] Cây rỗng → `[]`?